# Tên công ty

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://infodoanhnghiep.com"
LIST_URL = "https://infodoanhnghiep.com/Hai-Phong/"
OUTPUT_FILE = "companies_full.json"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

MAX_WORKERS = 10
RETRIES = 3
TIMEOUT = 10
SLEEP_BETWEEN_PAGES = 0.3

# =========================
# UTILS
# =========================

def safe_request(url):
    for attempt in range(RETRIES):
        try:
            res = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if res.status_code == 200:
                return res
        except Exception:
            time.sleep(1)
    return None


def save_json_incremental(file, data):
    # append từng record (không ghi lại toàn bộ)
    with open(file, "a", encoding="utf-8") as f:
        f.write(json.dumps(data, ensure_ascii=False) + "\n")


# =========================
# PARSE LIST
# =========================

def get_company_list(url):
    res = safe_request(url)
    if not res:
        return []

    soup = BeautifulSoup(res.text, "html.parser")
    items = soup.find_all("div", class_="company-item")

    companies = []

    for item in items:
        name_tag = item.find("h3")
        name = name_tag.get_text(strip=True) if name_tag else "N/A"

        address_tag = item.find("p")
        address = address_tag.get_text(strip=True) if address_tag else "N/A"
        address = re.sub(r"\d{2}/\d{2}/\d{4}", "", address).strip()

        link_tag = item.find("a")
        link = None
        if link_tag:
            href = link_tag.get("href")
            if href and href.startswith("http"):
                link = href
            elif href:
                link = BASE_URL + href

        companies.append({
            "name": name,
            "address": address,
            "link": link
        })

    return companies


# =========================
# GET MST
# =========================

def get_mst(url):
    res = safe_request(url)
    if not res:
        return None

    text = res.text
    match = re.search(r"Mã số thuế[^0-9]*(\d{10,13})", text)
    return match.group(1) if match else None


# =========================
# WORKER
# =========================

def process_company(c):
    mst = None
    if c["link"]:
        mst = get_mst(c["link"])

    result = {
        "name": c["name"],
        "address": c["address"],
        "link": c["link"],
        "mst": mst
    }

    print(f"✔ {c['name']} => {mst}", flush=True)

    # save ngay khi xong
    save_json_incremental(OUTPUT_FILE, result)

    return result


# =========================
# PIPELINE REAL-TIME
# =========================

def crawl_pipeline(max_pages=None):
    page = 1

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []

        while True:
            if max_pages and page > max_pages:
                print("⛔ Stop by max_pages")
                break

            url = LIST_URL if page == 1 else f"{LIST_URL}trang-{page}/"
            print(f"📄 Page {page}: {url}")

            companies = get_company_list(url)

            if not companies:
                print("⛔ Hết dữ liệu")
                break

            for c in companies:
                futures.append(executor.submit(process_company, c))

            page += 1
            time.sleep(SLEEP_BETWEEN_PAGES)

        # chờ toàn bộ thread xong
        for future in as_completed(futures):
            future.result()

    print("🎉 DONE PIPELINE")


# =========================
# RUN
# =========================

if __name__ == "__main__":
    # crawl 10 page
    crawl_pipeline(max_pages=10)


📄 Page 1: https://infodoanhnghiep.com/Hai-Phong/
📄 Page 2: https://infodoanhnghiep.com/Hai-Phong/trang-2/
✔ CÔNG TY CỔ PHẦN XUẤT NHẬP KHẨU HÓA CHẤT VÀ THIẾT BỊ AN PHÁT => 0202299209
✔ CÔNG TY TNHH DỊCH VỤ THỰC PHẨM TAVISON => 0202298011
✔ CÔNG TY TNHH XUẤT NHẬP KHẨU VINACO => 0801458070
✔ CÔNG TY TNHH VN-HOME => 0801459243
✔ CÔNG TY TNHH THƯƠNG MẠI VÀ DỊCH VỤ THỊNH HƯNG VN => 0202300535
✔ CÔNG TY TNHH PHÁT TRIỂN AUTO TUẤN BẢO => 0202299858
✔ CÔNG TY TNHH NEWLIFE QUÝ NHÂN => 0202295469
✔ CÔNG TY TNHH CÔNG NGHỆ QUANG ĐIỆN XINHAO VIỆT NAM (MIỀN BẮC VIỆT NAM) => 0202298036
✔ CÔNG TY TNHH ĐẠI PHÁT TRADE => 0202295451
✔ CÔNG TY TNHH B.E.S.T 68 PHƯƠNG NGA => 0202298117
✔ CÔNG TY TNHH TMDV HUY DUNG => 0202295349
✔ CÔNG TY TNHH SẢN XUẤT DỊCH VỤ VÀ THƯƠNG MẠI KIẾN AN FOODS => 0202298029
✔ CÔNG TY TNHH TÂN SINH INVESTMENT CONSULTANTS => 0202295388
✔ CÔNG TY TNHH THƯƠNG HIỆU THỜI TRANG VN => 0202295356
✔ CÔNG TY TNHH TM VÀ DV VẬN TẢI QUANG MINH => 0202295370
✔ CÔNG TY TNHH GIẢI PHÁP PRIEFERT VIỆT 

# 2. Mã số thuế


In [ ]:
!wget https://masothue.com/0801459243-cong-ty-tnhh-vn-home -O test.html

In [2]:
import subprocess
from bs4 import BeautifulSoup
import json
import re
import unicodedata
import time
import random


# ========================
# FETCH HTML (wget)
# ========================
def fetch_html(url):
    result = subprocess.run(
        ["wget", "-qO-", url],
        capture_output=True,
        text=True
    )
    return result.stdout


# ========================
# RETRY (anti-block)
# ========================
def fetch_html_retry(url, retries=3):
    for i in range(retries):
        html = fetch_html(url)

        # check bị block
        if "Enable JavaScript and cookies to continue" not in html:
            return html

        print(f"⚠️ Bị block, retry lần {i+1}")
        time.sleep(random.uniform(2, 5))

    return html


# ========================
# SLUGIFY
# ========================
def slugify(text):
    text = text.lower()

    text = text.replace("đ", "d")

    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')

    text = re.sub(r'[^a-z0-9]+', '-', text)

    return text.strip('-')


# ========================
# BUILD URL
# ========================
def build_url(mst, name):
    slug = slugify(name)
    return f"https://masothue.com/{mst}-{slug}"


# ========================
# PARSE COMPANY
# ========================
def parse_company(mst, name):
    url = build_url(mst, name)
    html = fetch_html_retry(url)

    soup = BeautifulSoup(html, "html.parser")
    name_tag = soup.select_one("thead span.copy")
    # print("name_tag: ", name_tag)

    if not name_tag:
        print("⚠️ Slug sai → fallback MST")
        print("url old: ", url)
        url = f"https://masothue.com/{mst}"
        html = fetch_html_retry(url)
        soup = BeautifulSoup(html, "html.parser")
        name_tag = soup.select_one("thead span.copy")


    data = {
        "mst": mst,
        "url": url,
        "name": None,
        "address": None,
        "status": None,
        "business": {
            "main": None,
            "detail": None
        }
    }


    if name_tag:
        data["name"] = name_tag.get_text(strip=True)


    rows = soup.select("tbody tr")

    for row in rows:
        cols = row.find_all("td")
        if len(cols) < 2:
            continue

        label = cols[0].get_text(strip=True)
        value_td = cols[1]


        if label == "Địa chỉ":
            data["address"] = value_td.get_text(" ", strip=True)

        elif "Tình trạng" in label:
            data["status"] = value_td.get_text(strip=True)


        elif "Ngành nghề chính" in label:

            a_tag = value_td.find("a")
            if a_tag:
                data["business"]["main"] = a_tag.get_text(strip=True)

            full_text = value_td.get_text(" ", strip=True)
            if "Chi tiết:" in full_text:
                data["business"]["detail"] = full_text.split("Chi tiết:")[-1].strip()

    return data


# ========================
# MAIN
# ========================
companies = []
with open("companies_full.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            companies.append(json.loads(line))

results = []

for i, c in enumerate(companies, 1):
    if c.get("mst") and c.get("name"):
        print(f"\n------ {i} -------\n")

        data = parse_company(c["mst"], c["name"])

        print("URL:", data["url"])
        print("Tên:", data["name"])
        print("Ngành:", data["business"]["main"])

        results.append(data)

        # delay tránh block
        time.sleep(random.uniform(1, 3))


# ========================
# SAVE JSON
# ========================
with open("final_data.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)


------ 1 -------

URL: https://masothue.com/0202299209-cong-ty-co-phan-xuat-nhap-khau-hoa-chat-va-thiet-bi-an-phat
Tên: CÔNG TY CỔ PHẦN XUẤT NHẬP KHẨU HÓA CHẤT VÀ THIẾT BỊ AN PHÁT
Ngành: Bán buôn chuyên doanh khác chưa được phân vào đâu

------ 2 -------

URL: https://masothue.com/0202298011-cong-ty-tnhh-dich-vu-thuc-pham-tavison
Tên: CÔNG TY TNHH DỊCH VỤ THỰC PHẨM TAVISON
Ngành: Dịch vụ ăn uống khác

------ 3 -------

URL: https://masothue.com/0801458070-cong-ty-tnhh-xuat-nhap-khau-vinaco
Tên: CÔNG TY TNHH XUẤT NHẬP KHẨU VINACO
Ngành: Hoạt động dịch vụ hỗ trợ kinh doanh khác còn lại chưa được phân vào đâu

------ 4 -------

URL: https://masothue.com/0801459243-cong-ty-tnhh-vn-home
Tên: CÔNG TY TNHH VN-HOME
Ngành: Hoạt động thiết kế chuyên dụng

------ 5 -------

URL: https://masothue.com/0202299858-cong-ty-tnhh-phat-trien-auto-tuan-bao
Tên: CÔNG TY TNHH PHÁT TRIỂN AUTO TUẤN BẢO
Ngành: Bán phụ tùng và các bộ phận phụ trợ của ô tô và xe có động cơ khác

------ 6 -------

URL: https://m

# 3. Phán đoán khả năng tuyển dụng 1 ngành nghề cụ thể


In [3]:
import json

# ===== KEYWORDS =====
IT_KEYWORDS = {
    "công nghệ": 40,
    "phần mềm": 40,
    "hệ thống": 30,
    "dịch vụ": 10,
    "media": 10,
    "tư vấn": 10,
    "bán lẻ": 5,
    "bán buôn": 5,
}

LOGISTICS_KEYWORDS = {
    "vận tải": 50,
    "xuất khẩu": 40,
    "nhập khẩu": 40,
    "xuất nhập khẩu": 50,
    "giao nhận": 40,
    "hải quan": 40,
    "kho": 30,
}

NEGATIVE_IT = {
    "nhà hàng": -30,
    "ăn uống": -30,
    "sản xuất": -20,
    "may": -15,
}

# ===== SCORING FUNCTION =====
def calculate_score(text, keywords):
    score = 0
    matched = []

    for k, v in keywords.items():
        if k in text:
            score += v
            matched.append(k)

    return score, matched


def calculate_negative(text, keywords):
    score = 0
    matched = []

    for k, v in keywords.items():
        if k in text:
            score += v
            matched.append(k)

    return score, matched


def analyze_company(company):
    text = (
        (company.get("business", {}).get("main") or "") + " " +
        (company.get("business", {}).get("detail") or "")
    ).lower()

    it_score, it_matched = calculate_score(text, IT_KEYWORDS)
    log_score, log_matched = calculate_score(text, LOGISTICS_KEYWORDS)
    neg_score, neg_matched = calculate_negative(text, NEGATIVE_IT)

    it_score += neg_score

    # normalize
    it_score = max(0, min(it_score, 100))
    log_score = max(0, min(log_score, 100))

    return {
        "name": company["name"],
        "it_score": it_score,
        "logistics_score": log_score,
        "it_keywords": it_matched,
        "logistics_keywords": log_matched,
        "negative_keywords": neg_matched
    }


# ===== MAIN =====
def analyze_all(data):
    results = [analyze_company(c) for c in data]

    # sort theo IT score
    results.sort(key=lambda x: x["it_score"], reverse=True)

    return results


# ===== RUN =====
if __name__ == "__main__":
    with open("final_data.json", "r", encoding="utf-8") as f:
      data = json.load(f)

    results = analyze_all(data)

    for r in results:
        print("=" * 50)
        print(f"🏢 {r['name']}")
        print(f"👉 IT score: {r['it_score']}")
        print(f"👉 Logistics score: {r['logistics_score']}")
        print(f"   + IT keywords: {r['it_keywords']}")
        print(f"   + Logistics keywords: {r['logistics_keywords']}")
        print(f"   - Negative: {r['negative_keywords']}")

🏢 CÔNG TY TNHH CAN THIỆP SỚM VÂN TRANG
👉 IT score: 50
👉 Logistics score: 0
   + IT keywords: ['hệ thống', 'dịch vụ', 'tư vấn']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNHH TRUSTIVE
👉 IT score: 45
👉 Logistics score: 0
   + IT keywords: ['phần mềm', 'bán lẻ']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNHH TM LÊ THUẬN
👉 IT score: 45
👉 Logistics score: 0
   + IT keywords: ['phần mềm', 'bán buôn']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNHH THƯƠNG MẠI VÀ PHÁT TRIỂN CÔNG NGHỆ LÂM DŨNG
👉 IT score: 40
👉 Logistics score: 0
   + IT keywords: ['phần mềm']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNHH KỸ THUẬT HỆ THỐNG CÔNG TRÌNH GUANG YI HENG VIỆT NAM
👉 IT score: 30
👉 Logistics score: 0
   + IT keywords: ['hệ thống']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNHH THƯƠNG MẠI VÀ XÂY DỰNG CƠ ĐIỆN 393
👉 IT score: 30
👉 Logistics score: 0
   + IT keywords: ['hệ thống']
   + Logistics keywords: []
   - Negative: []
🏢 CÔNG TY TNH